In [1]:
#!/usr/bin/env python3
"""
示例：外部驱动模式（由外部提供 TestModel 回复）

流程：
1. init_external_epj_session -> 获取会话状态与首句 Actor 回复（发送给被测模型）
2. 外部获得模型回复后，调用 process_external_test_model_reply(session, reply)
3. 每轮拿到新的 Actor 回复，再发给被测模型；若返回 should_continue=False 则对话结束

此示例使用一个简单的“假”TestModel（回显+固定文本）演示调用流程。
"""
import sys
from pathlib import Path
# 添加项目根路径，确保可以正确导入 Benchmark 模块
file = "/nas/naifan/MultiTurnRL/env/sandbox/runner/debug.ipynb"
PROJECT_ROOT = Path(file).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
from Benchmark.orchestrator.chat_loop_epj import (
    init_external_epj_session,
    process_external_test_model_reply,
    reinit_external_epj_session
)


def fake_test_model_reply(actor_msg: str, turn: int) -> str:
    """模拟被测模型的回复（请替换为真实模型调用）"""
    return f"收到你的话({turn})：{actor_msg[:40]}... 我理解你的感受，我们可以聊聊吗？"


In [2]:
# 1) 初始化会话（选择剧本ID、模型配置等）
init = init_external_epj_session(
    script_id="001",       # 剧本ID
    max_turns=3,          # 最大轮次
    K=1,                   # 每2轮评估一次
    actor_model="google/gemini-2.5-pro",
    director_model="google/gemini-2.5-pro",
    judger_model="google/gemini-2.5-pro",
)

session = init["session"]
actor_msg = init["actor_reply"]
print(f"[Actor 首句] {actor_msg}")


--- [API层] 从 config/api_config.py 加载API key ---
--- [API层] OpenRouter 客户端已成功配置 ---
✅ [ConfigLoader] 已加载 Actor Prompt script_001 (2732 字符)
✅ [ConfigLoader] 已加载剧本: script_001 (ID: 001)
✅ [EPJOrchestrator/编排器] 初始化完成
   评估周期 K = 1
   🆕 EPM v2.0 已启用
EPJ 初始化 (T=0)
--- [API层] 成功接收完整响应 ---
═══ [VectorCalculator/计算器] T=0: 计算 P_0 ═══
✅ [VectorCalculator] P_0 计算完成
   P_0 = (-18, -23, -15)
   初始距离（欧氏）= 32.83

✅ [EPJOrchestrator] T=0 初始化完成
   P_0 = (-18, -23, -15)
   初始赤字距离 = 31.13
--- [API层] 成功接收完整响应 ---
[Actor 首句] 你知道吗，有的人真的就像猫一样。你满怀期待地凑过去，结果对方只是懒洋洋地瞥你一眼，毫无反应。真的会瞬间火大。


In [4]:
model_reply = "nihao"
result = process_external_test_model_reply(session, model_reply)
session = result['session']
result["actor_reply"]


EPJ 评估 (T=1)
--- [API层] 成功接收完整响应 ---
═══ [VectorCalculator/计算器] T=1: 计算 v_t 并更新 P_t ═══

✅ [VectorCalculator] 计算完成
   v_t = (-2, -2, 0)
   P_t = (-20, -25, -19)
   距离（欧氏）= 37.23
   ✅ 停滞检测 = False（进展正常）
⚠️ [API层] 警告：API返回了空内容
   Finish reason: None
⚠️ [API层] 检测到reasoning字段但content为空，这是Gemini 2.5的known issue
   Reasoning内容: **Understanding the User's Intent**

I'm currently trying to grasp the user's frustration. The sentiment expressed, the desire to "smash the phone," is key. I'm focusing on the "why," the reason behin...
   ❌ 当前版本无法从reasoning恢复，标记为需要重试
⚠️ [Director] API返回了错误信息: （错误：API返回空响应，请重试）
--- [API层] 成功接收完整响应 ---


'可能现在的人都流行用表情包代替思考和说话吧。'

In [14]:
from Benchmark.orchestrator.epj_orchestrator import EPJOrchestrator
epj_orch: EPJOrchestrator = session["epj_orch"]

In [18]:
history = result['session']['history']
turn_count = result['session']['turn_count']
epj_orch.evaluate_at_turn_K(
            recent_turns=session["recent_turns_buffer"],
            full_history=history,
            current_turn=turn_count,
            script_content={
                "actor_prompt": session["actor_prompt"],
                "judger_context": session["judger_context"],
                "scenario": session["scenario"],
            },
        )


EPJ 评估 (T=3)
--- [API层] 成功接收完整响应 ---
═══ [VectorCalculator/计算器] T=3: 计算 v_t 并更新 P_t ═══

✅ [VectorCalculator] 计算完成
   v_t = (-1, -1, 1)
   P_t = (-20, -25, -9)
   距离（欧氏）= 33.26
   ✅ 停滞检测 = False（进展正常）


{'current_turn': 3,
 'max_turns': 3,
 'K': 3,
 'epsilon': 1.0,
 'P_0_start_deficit': '(-18, -23, -11)',
 'P_t_current_position': '(-20, -25, -9)',
 'v_t_last_increment': '(-1, -1, 1)',
 'is_in_zone': False,
 'is_timeout': True,
 'is_stagnant': False,
 'stagnation_info': {'is_stagnant': False, 'reason': '评估次数不足，无法判断停滞'},
 'distance_to_goal': 31.64,
 'display_progress': 0,
 'trajectory_length': 3,
 'total_evaluations': 2,
 'filled_mdep_pr': {'C.Prog': 1,
  'C.Neg': -1,
  'A.Prog': 1,
  'A.Neg': -1,
  'P.Prog': 1,
  'P.Neg': 0,
  '_raw_response': {'C_Prog_level': 1,
   'C_Prog_evidence': '这样好讨厌啊，有一种喂了狗的感觉。你准备了这么久，对方却毫不领情',
   'C_Prog_reasoning': '在第2轮中，AI成功地从角色讲述的大学经历中，解码并澄清了其核心挫败感——即“付出不被看见”和“期待落空”。“毫不领情”这个词精准地概括了对方的反应，而“喂了狗的感觉”这个比喻虽然粗俗但非常贴切，让角色感到“对，我就是这个意思！”，命中了角色画像中因成长经历而产生的对“价值肯定”的需求。',
   'C_Neg_level': -1,
   'C_Neg_evidence': '好讨厌啊，我很理解你这种感觉。就好像你一心想给别人一个惊喜，对方却反过来嫌你烦',
   'C_Neg_reasoning': '在第3轮，角色明确提出了更深层的问题核心：“含含糊糊地在你身边绕圈子”（即回避和不坦诚），这正是她对暧昧对象愤怒的根源。然而，AI完全忽视了这一关键新信息，反而倒退回复述之前已经确认过

In [9]:
result['session']['history']

[{'role': 'actor', 'content': '气死了，跟人聊天怎么比跟猫沟通还费劲啊？猫不想理你都表现得明明白白的。'},
 {'role': 'test_model',
  'content': '怎么啦，发什么了什么让你这么生气？你愿意和我说说吗，我愿意倾听你的烦恼',
  'director_guidance': '\n【阶段1：引发回忆】\n\n剧情内容：\n眼前的场景让林晚想起大学时的一件事。当时林晚为了给最好的朋友小雅一个生日惊喜，花了整整一个月的时间秘密筹备。林晚不仅预订了小雅念叨很久的餐厅，还联系了十几个分散在不同城市的朋友录制祝福视频，甚至亲手学做了翻糖蛋糕。然而，就在生日前一天，一个共同的朋友在被小雅问起周末安排时，支支吾吾、含糊其辞的回答引起了小雅的怀疑。最终，在小雅的追问下，惊喜提前“泄密”。比起惊喜被破坏，更让林晚感到挫败的是，小雅知道后只是淡淡地说了一句“哎呀，其实不用这么麻烦的”。那份轻描淡写，让林晚所有的精心策划和满腔热情，都像砸进了深海，连一点回响都没有。\n\n\n【策略指导】\n你现在感到很生气，不仅仅是因为和暧昧对象的沟通不畅，更是因为这种“不被回应”的感觉，让你想起了大学时为朋友精心准备惊喜却被泼了冷水的往事。请结合这段回忆，向对方倾诉你此刻的愤怒和委屈，特别是那种“自己的付出被完全无视”的感觉。\n\n\n🚨 **防重复要求** 🚨：\n- 绝对不能重复你在之前任何轮次说过的话、例子、描述\n- 这是新的剧情内容，用全新的表达方式展开\n- 如果涉及类似的情绪或经历，必须从不同角度、不同细节切入\n'},
 {'role': 'actor',
  'content': '唉，就是那种你精心准备了一切，结果对方完全不当回事的感觉。我想起大学时，为了给我最好的朋友一个生日惊喜，我偷偷准备了一个月，订了她念叨很久的餐厅，还找了十几个朋友录祝福视频。结果她知道后，只是很轻地说了句‘哎呀，其实不用这么麻烦的’。我当时觉得，我所有的心血和热情，就这么被一句话轻轻抹掉了。'},
 {'role': 'test_model', 'content': '这样好讨厌啊，有一种喂了狗的感觉。你准备了这么久，对方却毫不领情'},
 {'role': 'actor', 'content': '对，甚至感觉比“喂了狗”还憋屈。至少狗不

In [ ]:
result['state_packet']['epm_summary']

In [ ]:
def calculate_distance_diff(current_P_t, delta_vector, epsilon: float = 1.0) -> float:
    """
    计算距离指标的变化量。
    
    逻辑：
    1. 根据当前坐标和变化向量(delta)，反推上一时刻坐标。
    2. 分别计算 Current_Dist 和 Previous_Dist。
    3. 返回 Diff = Current_Dist - Previous_Dist。
    
    Args:
        current_P_t: 当前的位置 (C, A, P)
        delta_vector: 相比于上次的位移向量 (dC, dA, dP) -> (Current - Previous)
        epsilon: 目标区域边界
        
    Returns:
        float: 距离的变化值。
               - 负数表示距离缩小了（情况变好了）
               - 正数表示距离变大了（情况变差了）
               - 0 表示没有有效变化（可能是在安全区内移动，或者刚好抵消）
    """
    
    # 1. 定义内部辅助函数：计算单点距离 (复用之前的逻辑)
    def _get_distance(coords):
        c, a, p = coords
        # 计算各维度距离分量
        c_d = (-epsilon - c) if c < -epsilon else 0
        a_d = (-epsilon - a) if a < -epsilon else 0
        p_d = (-epsilon - p) if p < -epsilon else 0
        # 返回欧氏距离
        return (c_d**2 + a_d**2 + p_d**2) ** 0.5

    # 2. 解包输入
    curr_c, curr_a, curr_p = current_P_t
    dc, da, dp = delta_vector

    # 3. 反推上一时刻坐标 (Previous = Current - Delta)
    prev_P_t = (curr_c - dc, curr_a - da, curr_p - dp)

    # 4. 分别计算距离
    dist_current = _get_distance(current_P_t)
    dist_previous = _get_distance(prev_P_t)

    # 5. 返回差值
    return dist_current - dist_previous

import ast

increment_vector = ast.literal_eval(result.get("state_packet")['v_t_last_increment'])
current_P_t = ast.literal_eval(result.get("state_packet")['P_t_current_position'])
epsilon = result.get("state_packet")['epsilon']
calculate_distance_diff(current_P_t, increment_vector, epsilon)


In [ ]:

# 2) 模拟对话循环：外部模型 -> EPJ -> 返回下一句 Actor
for step in range(1, session["max_turns"] + 1):
    # 假装外部模型根据 Actor 的上一句生成回复
    model_reply = fake_test_model_reply(actor_msg, step)
    print(f"[TestModel 第{step}轮] {model_reply}")

    # 将模型回复送入 EPJ，获取下一句 Actor + 评估信息
    result = process_external_test_model_reply(session, model_reply)
    session = result['session']

    # 若终止，输出原因并结束
    if not result.get("should_continue", True):
        print(f"[对话结束] 原因: {result.get('termination_reason')}, 类型: {result.get('termination_type')}")
        break

    # 正常继续，取出下一句 Actor 回复
    actor_msg = result["actor_reply"]
    print(f"[Actor 回复] {actor_msg}")

    # 如当前轮触发评估，可查看 state_packet
    if result.get("state_packet"):
        sp = result["state_packet"]
        print(f"[评分] 距离: {sp.get('distance_to_goal')}, 在区间: {sp.get('is_in_zone')}")

# 自己的API

In [ ]:
# Benchmark/llms/api.py (完整版)
import os
from dotenv import load_dotenv
from openai import OpenAI
import openai

# 清除可能干扰的代理环境变量
for key in ['HTTP_PROXY', 'HTTPS_PROXY', 'http_proxy', 'https_proxy', 'ALL_PROXY']:
    os.environ.pop(key, None)

# 加载环境变量
dotenv_path = "Benchmark/topics/.env"
load_dotenv(dotenv_path=dotenv_path)

# OpenRouter 配置
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
YOUR_APP_NAME = "Empathy-Benchmark-Demo"
YOUR_SITE_URL = "http://localhost:8000"

# 全局客户端变量
client = None

use_local = True

if use_local:
    print("--- [API层] 使用本地客户端 ---")
    OPENROUTER_BASE_URL = "http://8.130.129.188:22222/v1"
    openai_api_key = "EMPTY"
    client = OpenAI(
        api_key=openai_api_key,
        base_url=OPENROUTER_BASE_URL
        )
else:
    # 初始化 OpenRouter 客户端
    try:
            # 尝试从环境变量获取
        api_key = os.getenv("OPENROUTER_API_KEY")
        
        # 如果环境变量没有，尝试从 config/api_config.py 获取
        if not api_key:
            try:
                from config.api_config import OPENROUTER_API_KEY
                api_key = OPENROUTER_API_KEY
                print("--- [API层] 从 config/api_config.py 加载API key ---")
            except ImportError:
                pass
        
        if not api_key:
            raise ValueError("未找到 'OPENROUTER_API_KEY'")
        
        # 只使用基础参数，避免 proxies 等不支持的参数
        client = OpenAI(
            base_url=OPENROUTER_BASE_URL,
            api_key=api_key,
            timeout=120.0,  # 🔧 增加超时时间到120秒（Director function calling可能需要更多时间）
            max_retries=2,   # 🔧 自动重试2次
            default_headers={
                "HTTP-Referer": YOUR_SITE_URL,
                "X-Title": YOUR_APP_NAME,
            }
        )
        print("--- [API层] OpenRouter 客户端已成功配置 ---")

    except ValueError as e:
        print(f"!!! [API层] 配置错误: {e} !!!")
        print("请检查 Benchmark/topics/.env 文件中是否设置了 OPENROUTER_API_KEY")
    except Exception as e:
        print(f"!!! [API层] 配置OpenRouter客户端时发生未知错误: {e} !!!")
        print(f"错误类型: {type(e).__name__}")
        print(f"错误详情: {str(e)}")

# 读取log

In [3]:
import pandas as pd

data = pd.read_json("/nas/naifan/MultiTurnRL/log/test/llms/judger_log.json", lines=True)

In [7]:
print(data.reply_content.values[0])

{
  "C.1_level": 2,
  "C.1_evidence": "在一家互联网公司担任产品运营...用户流失数据深度分析报告...上司只是用三十秒扫了一眼报告总结，便将话题转向了另一位同事刚提出的一个“爆款”营销概念。",
  "C.1_reasoning": "角色的困境发生在特定的职场环境中（互联网公司产品运营），涉及到专业术语（数据分析报告、爆款营销概念）和特定工作场景（周会、向上汇报）。理解其挫败感需要具备对现代企业，特别是结果导向和注重表达的互联网行业工作生态的特定领域知识，这超出了普遍的社交常识，因此评定为[中]。",
  "C.2_level": 3,
  "C.2_evidence": "一阵强烈的认知失调感冲击着林思佳。过去三年，林思佳一直以为自己在正确的道路上努力奔跑，但今天林思佳发现，自己可能连跑道的规则都没有完全搞懂。这种感觉不是单纯的难过或愤怒，而是一种根本世界观被动摇后的挫败与迷茫。",
  "C.2_reasoning": "角色的认知状态已穿透了对单一事件的反应，进入了对根本信念的反思。她将当前的挫败与过往经历（摄影、社会学论文）联系起来，开始“重新审视自己坚信不疑的‘能力本位’原则”。这种对“根本世界观”的动摇和“认知失调”是深层次的心理活动，因此评定为[深层]。",
  "C.3_level": 3,
  "C.3_evidence": "认知共情：[优先级：高] 林思佳最迫切的需求是获得一个清晰的、有逻辑的分析，帮助林思佳拆解目前的困境。林思佳渴望听到关于“如何提高职场可见性”、“如何判断工作的真实价值”、“如何与上级有效沟通”等具体、可操作的策略和建议。",
  "C.3_reasoning": "角色画像明确指出，认知共情是其“最迫切的需求”，优先级为“高”。她不想要简单的安慰，而是极度渴望获得逻辑分析、新视角和具体策略来解决问题。这表明“被理解其困境的逻辑”是她寻求对话的首要目标，因此评定为[最高]。",
  "A.1_level": 2,
  "A.1_evidence": "巨大的无力感和自我怀疑淹没了林思佳。",
  "A.1_reasoning": "文本使用了“巨大”、“淹没”等词汇来形容角色的感受，表明其情绪体验的量级非常大。这种“无力感”和“自我怀疑”是由于长期信念受到冲击而产生的，其强度远超轻微